# Brazilian E-Commerce Analytics
### Business Understanding & Initial Data Exploration

**Dataset:** Brazilian E-Commerce Public Dataset by Olist  
**Database:** PostgreSQL  
**Languages & Libraries:** Python, Pandas, SQL

---

## 1. Business Context

Olist is a Brazilian e-commerce marketplace that connects multiple sellers with customers through an online platform. Marketplace activity generates data across orders, customers, products, sellers, payments, reviews, and delivery operations.

These datasets provide different views of the same business. When analyzed together, they can help explain marketplace performance, customer purchasing behavior, product contribution, seller performance, and delivery efficiency.

This project uses the Olist Brazilian E-Commerce Public Dataset to explore these areas and develop SQL-based business analysis.

## 2. Business Problem

A large volume of transactional data does not automatically provide meaningful business insight. Data is distributed across related tables, recorded at different levels of detail, and may contain missing or inconsistent values.

The challenge is to understand the structure and quality of the data, combine related information at the correct level, and define reliable metrics before drawing business conclusions.

The goal of this project is to turn the raw marketplace data into a structured analytical foundation for evaluating business performance and identifying potential improvement areas.


## 3. Business Questions

The analysis is guided by the following questions.

### Marketplace Performance
- What are the marketplace's recorded revenue and order volumes?
- How do revenue and order volumes change over time?
- What is the average order value, and how should it be calculated?
- Which periods show the highest observed marketplace activity?

### Customer Behavior
- How many distinct customers are represented in the dataset?
- What proportion of customers place more than one order?
- How is customer value distributed?
- Which customer groups may benefit from retention or re-engagement strategies?

### Product Performance
- Which product categories contribute most to revenue?
- Which products have the highest sales volumes?
- Do the highest-volume products also generate the highest revenue?

### Seller Performance
- How is revenue distributed across sellers?
- How concentrated is revenue among the largest sellers?
- What do review scores associated with seller orders indicate?

### Delivery & Operations
- How long do delivered orders take to reach customers?
- What proportion of delivered orders arrive by the estimated delivery date?
- How does delivery performance vary by customer location?

### Revenue Reconciliation
- How do payment-based totals compare with order-item-based totals?
- Are any delivered orders missing payment records?
- How large are the order-level differences between these two revenue measures?

## 4. Analytical Objectives

1. Understand the structure and relationships of the Olist datasets.
2. Profile data completeness, uniqueness, and consistency.
3. Establish appropriate definitions and levels of detail for business metrics.
4. Analyze marketplace, customer, product, seller, and operational performance.
5. Apply Recency, Frequency, and Monetary (RFM) analysis for customer segmentation.
6. Reconcile payment-based and item-based revenue and investigate discrepancies.
7. Translate analytical results into clearly scoped business findings.


## 5. Dataset Understanding

The project uses the **Brazilian E-Commerce Public Dataset by Olist**, a collection of anonymized marketplace transactions from 2016–2018.

The project uses seven core relational datasets and two supporting datasets.

### Core Business Datasets

| Dataset | Records | Main business information |
|---|---:|---|
| Customers | 99,441 | Customer identifiers and geographic information |
| Orders | 99,441 | Order status and lifecycle timestamps |
| Order Items | 112,650 | Products, sellers, item prices, and freight |
| Order Payments | 103,886 | Payment methods, payment values, and installments |
| Products | 32,951 | Product attributes and category information |
| Sellers | 3,095 | Seller identifiers and geographic information |
| Reviews | 99,224 | Customer review scores and timestamps |

### Supporting Datasets

| Dataset | Records | Main business information |
|---|---:|---|
| Geolocation | 1,000,163 | ZIP-code-level geographic coordinates and location information |
| Category Translation | 71 | Portuguese-to-English product category mappings |


*Record counts are the counts observed during project profiling. They are not necessarily the number of unique business entities in every table.*

### 5.1 Table Relationships

The main relationships used in the analysis include:

- `orders.customer_id` links an order to its customer record.
- `order_items.order_id` links order items to orders.
- `order_payments.order_id` links payment records to orders.
- `order_reviews.order_id` links reviews to orders.
- `order_items.product_id` links order items to products.
- `order_items.seller_id` links order items to sellers.

`customer_unique_id` identifies a customer across potentially multiple order-specific customer records. It is therefore used for distinct-customer and repeat-purchase analysis rather than counting `customer_id`.

### 5.2 Why the Level of Detail Matters

The tables do not all contain one row per order. For example, an order can contain multiple order items and multiple payment records. Joining these tables without first considering their level of detail can multiply rows and inflate revenue or other aggregated metrics.

The SQL analysis therefore treats order-level, item-level, payment-level, and customer-level calculations separately where required.


## 6. Initial Data Exploration

This section provides a reproducible starting point for inspecting the raw CSV files with Python and Pandas. The detailed profiling and validation work is also documented in the project's `etl/` scripts and SQL validation file.

### 6.1 Locate the Raw Dataset Files

The notebook is stored in the `notebooks/` directory. The following cell checks the expected raw-data folder and lists available CSV files.



In [1]:
from pathlib import Path

import pandas as pd

# Support running the notebook from either the project root or notebooks/ folder.
candidate_dirs = [Path("data/raw"), Path("../data/raw")]
raw_data_dir = next((path for path in candidate_dirs if path.exists()), None)

if raw_data_dir is None:
    raise FileNotFoundError(
        "Could not find the raw data folder. Expected data/raw or ../data/raw. "
        "Run this notebook from the project root or notebooks directory."
    )

csv_files = sorted(raw_data_dir.glob("*.csv"))

if not csv_files:
    raise FileNotFoundError(f"No CSV files found in: {raw_data_dir.resolve()}")

print(f"Raw data directory: {raw_data_dir.resolve()}")
print(f"CSV files found: {len(csv_files)}")

for file_path in csv_files:
    print(f"- {file_path.name}")


Raw data directory: C:\Users\Jahnavi P\Desktop\Projects\olist-analytics\data\raw
CSV files found: 9
- olist_customers_dataset.csv
- olist_geolocation_dataset.csv
- olist_order_items_dataset.csv
- olist_order_payments_dataset.csv
- olist_order_reviews_dataset.csv
- olist_orders_dataset.csv
- olist_products_dataset.csv
- olist_sellers_dataset.csv
- product_category_name_translation.csv


### 6.2 Inspect Dataset Dimensions

Load the available CSV files and review their row and column counts. This is an initial structural check; detailed validation is handled in the ETL and SQL workflows.


In [2]:
# Load each CSV into a dictionary using its filename (without extension) as the key.
dataframes = {
    file_path.stem: pd.read_csv(file_path, low_memory=False)
    for file_path in csv_files
}

dataset_overview = pd.DataFrame(
    [
        {
            "dataset": name,
            "rows": df.shape[0],
            "columns": df.shape[1]
        }
        for name, df in dataframes.items()
    ]
).sort_values("dataset").reset_index(drop=True)

display(dataset_overview)


,dataset,rows,columns
0,olist_customers_dataset,99441,5
1,olist_geolocation_dataset,1000163,5
2,olist_order_items_dataset,112650,7
3,olist_order_payments_dataset,103886,5
4,olist_order_reviews_dataset,99224,7
5,olist_orders_dataset,99441,8
6,olist_products_dataset,32951,9
7,olist_sellers_dataset,3095,4
8,product_category_name_translation,71,2


### 6.3 Inspect Column Names and Data Types

Review the available columns and inferred data types. This helps identify timestamp fields, identifiers, numeric measures, and categorical attributes that may require attention during transformation.


In [3]:
column_profile = []

for dataset_name, df in dataframes.items():
    for column in df.columns:
        column_profile.append(
            {
                "dataset": dataset_name,
                "column": column,
                "data_type": str(df[column].dtype),
                "non_null_count": int(df[column].notna().sum()),
                "null_count": int(df[column].isna().sum())
            }
        )

column_profile_df = pd.DataFrame(column_profile)
display(column_profile_df)


,dataset,column,data_type,non_null_count,null_count
0,olist_customers_dataset,customer_id,str,99441,0
1,olist_customers_dataset,customer_unique_id,str,99441,0
2,olist_customers_dataset,customer_zip_code_prefix,int64,99441,0
3,olist_customers_dataset,customer_city,str,99441,0
4,olist_customers_dataset,customer_state,str,99441,0
5,olist_geolocation_dataset,geolocation_zip_code_prefix,int64,1000163,0
6,olist_geolocation_dataset,geolocation_lat,float64,1000163,0
7,olist_geolocation_dataset,geolocation_lng,float64,1000163,0
8,olist_geolocation_dataset,geolocation_city,str,1000163,0
9,olist_geolocation_dataset,geolocation_state,str,1000163,0


### 6.4 Review Missing Values

Missing values should be interpreted in business context. For example, a missing delivery timestamp may be expected for an order that has not been delivered, whereas a missing key identifier may affect the ability to join or analyze records.

The following summary identifies columns with missing values; it does not automatically classify every missing value as an error.


In [4]:
missing_value_summary = (
    column_profile_df.loc[column_profile_df["null_count"] > 0]
    .sort_values(["dataset", "null_count"], ascending=[True, False])
    .reset_index(drop=True)
)

display(missing_value_summary)


,dataset,column,data_type,non_null_count,null_count
0,olist_order_reviews_dataset,review_comment_title,str,11568,87656
1,olist_order_reviews_dataset,review_comment_message,str,40977,58247
2,olist_orders_dataset,order_delivered_customer_date,str,96476,2965
3,olist_orders_dataset,order_delivered_carrier_date,str,97658,1783
4,olist_orders_dataset,order_approved_at,str,99281,160
5,olist_products_dataset,product_category_name,str,32341,610
6,olist_products_dataset,product_name_lenght,float64,32341,610
7,olist_products_dataset,product_description_lenght,float64,32341,610
8,olist_products_dataset,product_photos_qty,float64,32341,610
9,olist_products_dataset,product_weight_g,float64,32949,2


### Interpretation

The missing-value analysis identified gaps across several datasets.

Key observations include:

- Review comments are frequently missing, which does not necessarily indicate a data-quality issue because customers may submit a score without writing a comment.
- Missing delivery timestamps require consideration of order status before calculating delivery performance.
- Missing product attributes may affect product-level analysis and should be handled carefully.
- Missing approval timestamps should be considered when analyzing the order lifecycle.

Missing values must therefore be interpreted in the context of each business process rather than automatically removed.

### 6.5 Inspect Key Identifiers

The following cell checks selected identifiers where the corresponding columns are present. These checks are exploratory: composite keys and referential integrity are validated more fully in the dedicated ETL and SQL validation scripts.


In [5]:
key_columns = {
    "olist_orders_dataset": ["order_id"],
    "olist_customers_dataset": ["customer_id", "customer_unique_id"],
    "olist_order_items_dataset": ["order_id", "order_item_id"],
    "olist_order_payments_dataset": ["order_id", "payment_sequential"],
    "olist_products_dataset": ["product_id"],
    "olist_sellers_dataset": ["seller_id"],
    "olist_order_reviews_dataset": ["review_id", "order_id"]
}

key_profile = []

for dataset_name, columns in key_columns.items():

    df = dataframes[dataset_name]

    for column in columns:
        key_profile.append({
            "dataset": dataset_name,
            "column": column,
            "non_null_values": int(df[column].notna().sum()),
            "distinct_values": int(df[column].nunique(dropna=True))
        })

display(pd.DataFrame(key_profile))

,dataset,column,non_null_values,distinct_values
0,olist_orders_dataset,order_id,99441,99441
1,olist_customers_dataset,customer_id,99441,99441
2,olist_customers_dataset,customer_unique_id,99441,96096
3,olist_order_items_dataset,order_id,112650,98666
4,olist_order_items_dataset,order_item_id,112650,21
5,olist_order_payments_dataset,order_id,103886,99440
6,olist_order_payments_dataset,payment_sequential,103886,29
7,olist_products_dataset,product_id,32951,32951
8,olist_sellers_dataset,seller_id,3095,3095
9,olist_order_reviews_dataset,review_id,99224,98410


### Interpretation

The identifier profiling revealed several important patterns:

- The orders, customers, products, and sellers datasets contain unique primary identifiers.
- The 99,441 customer records represent 96,096 distinct customers based on `customer_unique_id`.
- The order items dataset contains multiple records per order because an order can include multiple products.
- The `order_id` can appear multiple times in the payments dataset because an order may have multiple payment records. Payment sequence numbers are also reused across different orders.
- Review IDs and order IDs can repeat individually, requiring a composite-key check.

These observations helped determine the appropriate identifiers and aggregation levels for subsequent analysis.
These checks compare non-null and distinct counts; they do not independently establish referential integrity across tables.

### 6.6 Composite Key Validation

Some datasets contain multiple records for the same order. To establish record-level uniqueness, composite-key checks were performed using the relevant identifier combinations.

The validation checks whether each selected combination uniquely identifies a record within its dataset.

In [6]:
composite_keys = {
    "olist_order_items_dataset": ["order_id", "order_item_id"],
    "olist_order_payments_dataset": ["order_id", "payment_sequential"],
    "olist_order_reviews_dataset": ["review_id", "order_id"]
}

composite_key_results = []

for dataset_name, columns in composite_keys.items():

    df = dataframes[dataset_name]

    duplicate_count = int(df.duplicated(subset=columns).sum())

    composite_key_results.append({
        "dataset": dataset_name,
        "composite_key": " + ".join(columns),
        "total_records": len(df),
        "duplicate_combinations": duplicate_count,
        "is_unique": duplicate_count == 0
    })

display(pd.DataFrame(composite_key_results))

,dataset,composite_key,total_records,duplicate_combinations,is_unique
0,olist_order_items_dataset,order_id + order_item_id,112650,0,True
1,olist_order_payments_dataset,order_id + payment_sequential,103886,0,True
2,olist_order_reviews_dataset,review_id + order_id,99224,0,True


### Interpretation

Certain datasets contain multiple records for the same order. Therefore, individual columns are not always sufficient to uniquely identify records.

Composite-key validation was performed on the following datasets:

| Dataset | Composite Key | Records | Duplicate Combinations |
|---|---|---:|---:|
| Order Items | order_id + order_item_id | 112,650 | 0 |
| Payments | order_id + payment_sequential | 103,886 | 0 |
| Reviews | review_id + order_id | 99,224 | 0 |

All three composite keys passed the uniqueness checks.

This confirms that the selected column combinations uniquely identify records within their respective datasets.

These findings establish record-level uniqueness for the selected combinations. They provide a foundation for subsequent joins and aggregations, alongside separate relationship and financial validation checks.

## 7. Data Quality Findings & Analytical Considerations

The project's profiling and validation work identified several observations that influence how the data should be analyzed.

| Dataset | Observation |
|---|---|
| Orders | Missing delivery timestamps and some chronological inconsistencies were identified. |
| Order Items | The composite key `(order_id, order_item_id)` was unique. Some records had zero freight values. |
| Products | Some product attributes and physical measurements were missing. |
| Customers | 99,441 customer records represented 96,096 distinct `customer_unique_id` values. |
| Payments | Nine zero-value payment records were identified. |
| Reviews | Repeated review IDs occurred across records, while `(review_id, order_id)` was unique. |
| Sellers | Seller records covered multiple Brazilian states. |

### 7.1 Important Analytical Considerations

- Use `customer_unique_id` for distinct-customer and repeat-purchase analysis.
- Account for missing delivery timestamps when defining delivery metrics.
- Aggregate payment records at the order level before comparing them with order-level item totals.
- Avoid joining one-to-many tables in a way that duplicates item or payment amounts.
- Clearly distinguish all-order marketplace metrics from delivered-order dashboard metrics.
- Treat incomplete observation periods carefully when comparing annual performance.

The detailed checks and their results are maintained in the `etl/` and `sql/` project files. The notebook's exploratory cells provide an additional way to inspect the raw data locally.

## 8. Analytical Direction & Transition to SQL

Based on the business questions, dataset relationships, and data quality considerations, the project was organized into the following analytical areas:

| Business area | Analytical focus |
|---|---|
| Marketplace | Revenue, order volume, AOV, and growth trends |
| Customers | Distinct customers, repeat purchases, spending behavior, and RFM segmentation |
| Products | Category revenue, product performance, and sales volume |
| Sellers | Revenue contribution and seller concentration |
| Operations | Delivery duration, on-time performance, and regional delays |
| Revenue Reconciliation | Comparison of payment-based and item-based revenue |

PostgreSQL was selected for the main business analysis because the project requires relational joins, aggregations, CTEs, window functions, and calculations across different levels of detail.

The detailed analysis is organized into modular SQL scripts in the `sql/` directory, progressing from data validation to marketplace, customer, product, operations, seller, dashboard-oriented, and revenue reconciliation analysis.

## 9. Scope & Limitations

- The dataset contains historical marketplace transactions from 2016–2018 and does not represent real-time business activity.
- The 2016 and 2018 observation periods are incomplete; annual comparisons require caution.
- Revenue based on payment records is not identical in definition to revenue calculated from order-item prices and freight.
- The cause of every order-level revenue discrepancy has not been established.
- Review scores associated with seller orders should not be interpreted as proof that the seller caused the customer's experience.

## 10. Conclusion

The business understanding and initial data exploration established the foundation for the Olist analytics project.

By understanding the marketplace context, defining business questions, inspecting the relational datasets, and identifying data quality considerations, the project established a structured approach to SQL-based analysis.

The subsequent SQL work evaluates marketplace performance, customer behavior, product contribution, seller concentration, delivery operations, and revenue consistency.
